# UFC Fight Scraper
Scrapes ufcstats.com for a lightweight dataset of UFC fight results.

**Columns collected:**
- `event_name`, `event_date`, `event_location`
- `r_fighter`, `b_fighter` (red/blue corner)
- `winner`, `is_title_bout`
- `weight_class`, `method`
- `finish_round`, `finish_time`

> **Note:** Prelim/Main Card/Main Event designation is not marked in the HTML on ufcstats.com.
> If you need that, it would require a different source (e.g. Tapology or the UFC website).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

## Step 1 — Collect all event URLs

In [ ]:
def get_event_urls():
    """Return a list of all completed UFC event page URLs."""
    url = 'http://www.ufcstats.com/statistics/events/completed?page=all'
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')

    event_urls = []
    for row in soup.find_all('tr', class_='b-statistics__table-row'):
        link = row.find('a', class_='b-link b-link_style_black')
        if link and link.get('href'):
            event_urls.append(link['href'])

    print(f'Found {len(event_urls)} events')
    return event_urls

event_urls = get_event_urls()

## Step 2 — Scrape each event page

In [ ]:
def parse_event_page(event_url):
    """
    Scrape one event page.
    Returns a list of fight dicts — one per fight on the card.
    """
    resp = requests.get(event_url)
    if resp.status_code != 200:
        print(f'  SKIP (HTTP {resp.status_code}): {event_url}')
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')

    # --- Event name ---
    title_el = soup.find('h2', class_='b-content__title')
    event_name = title_el.text.strip() if title_el else 'Unknown'

    # --- Event date and location ---
    event_date = None
    event_location = None
    for item in soup.find_all('li', class_='b-list__box-list-item'):
        label = item.find('i', class_='b-list__box-item-title')
        if not label:
            continue
        value = item.get_text(strip=True).replace(label.get_text(strip=True), '').strip()
        if 'Date' in label.text:
            event_date = value
        elif 'Location' in label.text:
            event_location = value

    # --- Fight rows ---
    fights = []
    rows = soup.find_all('tr', class_='b-fight-details__table-row__hover')

    for i, row in enumerate(rows):
        fight_url = row.get('data-link')
        cells = row.find_all('td', class_='b-fight-details__table-col')
        if not cells or not fight_url:
            continue

        def cell_texts(idx):
            """All <p> text values within a cell."""
            if idx >= len(cells):
                return []
            return [p.get_text(strip=True) for p in cells[idx].find_all('p')]

        # Column order: W/L | Fighter | Kd | Str | Td | Sub | Weight class | Method | Round | Time
        wl      = cell_texts(0)   # ['win', 'loss'] or ['loss', 'win'] etc.
        names   = cell_texts(1)   # [r_fighter, b_fighter]
        wclass  = cell_texts(6)   # ['Lightweight Bout'] — may include title text
        method  = cell_texts(7)
        rnd     = cell_texts(8)
        t       = cell_texts(9)

        r_fighter = names[0] if len(names) > 0 else None
        b_fighter = names[1] if len(names) > 1 else None

        # Winner: whichever fighter has 'win' status
        winner = None
        if wl:
            r_status = wl[0].lower() if len(wl) > 0 else ''
            b_status = wl[1].lower() if len(wl) > 1 else ''
            if 'win' in r_status:
                winner = r_fighter
            elif 'win' in b_status:
                winner = b_fighter
            elif 'draw' in r_status or 'nc' in r_status:
                winner = 'Draw/NC'

        # Title bout: check for a belt image in the weight class cell
        is_title_bout = 1 if cells[6].find('img') else 0

        # Weight class: strip " Bout" suffix and title prefix
        wc_text = wclass[0] if wclass else ''
        weight_class = wc_text.replace(' Bout', '').replace(' Title', '').replace('UFC ', '').strip()

        fights.append({
            'event_name':     event_name,
            'event_date':     event_date,
            'event_location': event_location,
            'r_fighter':      r_fighter,
            'b_fighter':      b_fighter,
            'winner':         winner,
            'is_title_bout':  is_title_bout,
            'weight_class':   weight_class,
            'method':         method[0] if method else None,
            'finish_round':   rnd[0] if rnd else None,
            'finish_time':    t[0] if t else None,
            'fight_url':      fight_url,
        })

    return fights

## Step 3 — Run the scraper
Adjust `event_urls[start:end]` to scrape a subset, or leave as-is for all events.

In [ ]:
all_fights = []

for i, url in enumerate(event_urls[:10]):  # change to event_urls[:10] to test
    print(f'[{i+1}/{len(event_urls)}] {url}')
    fights = parse_event_page(url)
    all_fights.extend(fights)
    time.sleep(0.5)  # be polite

print(f'\nTotal fights scraped: {len(all_fights)}')

## Step 4 — Save to CSV

In [ ]:
df = pd.DataFrame(all_fights)
df.to_csv('ufc_fights.csv', index=False)
print(f'Saved {len(df)} fights to ufc_fights.csv')
df.head(10)